In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import spacy

In [2]:
df = pd.read_csv("../data/raw_hr_data.csv")
pd.set_option("display.max_colwidth", None)
df.head()

,id,job_title,location,connection,fit
0,1,2019 C.T. Bauer College of Business Graduate (Magna Cum Laude) and aspiring Human Resources professional,"Houston, Texas",85,NaN
1,2,Native English Teacher at EPIK (English Program in Korea),Kanada,500+,NaN
2,3,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,NaN
3,4,People Development Coordinator at Ryan,"Denton, Texas",500+,NaN
4,5,Advisory Board Member at Celal Bayar University,"İzmir, Türkiye",500+,NaN


In [3]:
df = df.drop("fit",axis=1)
df['connection'] = df['connection'].replace("500+ ","500")
df = df.drop_duplicates(subset=['job_title'])
# looking for non english alphabets using regex 
df[df["location"].str.contains(r"[^\x00-\x7F]", regex=True, na=False)]

,id,job_title,location,connection
4,5,Advisory Board Member at Celal Bayar University,"İzmir, Türkiye",500
77,78,Human Resources Generalist at Schwan's,Amerika Birleşik Devletleri,500
93,94,Seeking Human Resources Opportunities. Open to travel and relocation.,Amerika Birleşik Devletleri,415


In [4]:
df['location'] = df['location'].str.replace("ü","u")
df['location'] = df['location'].str.replace("ş","s")
df['location'] = df['location'].str.replace("İ","I")

In [5]:
df["all_data"] = (
    df["job_title"].astype(str) + " " +
    df["location"].astype(str)
)

In [6]:
labels_hr_resources = [1,0,1,0,0,1,1,1,1,0,0,
                       1,1,1,1,1,1,1,1,1,1,1,
                       1,1,1,1,1,1,0,1,1,1,1,
                       0,0,0,1,1,0,0,0,0,1,0,
                       0,0,1,1,1,0,0,0]

df['relevance'] = labels_hr_resources

In [7]:
df.head(60)

,id,job_title,location,connection,all_data,relevance
0,1,2019 C.T. Bauer College of Business Graduate (Magna Cum Laude) and aspiring Human Resources professional,"Houston, Texas",85,"2019 C.T. Bauer College of Business Graduate (Magna Cum Laude) and aspiring Human Resources professional Houston, Texas",1
1,2,Native English Teacher at EPIK (English Program in Korea),Kanada,500,Native English Teacher at EPIK (English Program in Korea) Kanada,0
2,3,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,"Aspiring Human Resources Professional Raleigh-Durham, North Carolina Area",1
3,4,People Development Coordinator at Ryan,"Denton, Texas",500,"People Development Coordinator at Ryan Denton, Texas",0
4,5,Advisory Board Member at Celal Bayar University,"Izmir, Turkiye",500,"Advisory Board Member at Celal Bayar University Izmir, Turkiye",0
5,6,Aspiring Human Resources Specialist,Greater New York City Area,1,Aspiring Human Resources Specialist Greater New York City Area,1
6,7,Student at Humber College and Aspiring Human Resources Generalist,Kanada,61,Student at Humber College and Aspiring Human Resources Generalist Kanada,1
7,8,HR Senior Specialist,San Francisco Bay Area,500,HR Senior Specialist San Francisco Bay Area,1
9,10,Seeking Human Resources HRIS and Generalist Positions,Greater Philadelphia Area,500,Seeking Human Resources HRIS and Generalist Positions Greater Philadelphia Area,1
10,11,Student at Chapman University,"Lake Forest, California",2,"Student at Chapman University Lake Forest, California",0


In [8]:
#!python -m spacy download en_core_web_sm
nlp = spacy.load("en_core_web_sm")

The idea is to build a function and apply it to each row in the job title column. The function tokenizes the text, removes punctuation and stop words, converts the words to lowercase, lemmatizes them, and returns the cleaned words as a list for each row in the job_title and return the join str becasue each model will use its own tokenization method.

In [9]:
def clean_title(text):
    doc = nlp(text)

    clean_text = []

    for token in doc:
        if not token.is_stop and not token.is_punct:
            clean_text.append(token.lemma_.lower())

    return " ".join(clean_text)

In [10]:
df["clean_job_title"] = df["all_data"].apply(clean_title)

In [11]:
df = df.drop("all_data",axis=1)

In [12]:
df.to_csv("../data/clean_hr_data.csv", index=False)

In [16]:
df.columns

Index(['id', 'job_title', 'location', 'connection', 'relevance',
       'clean_job_title'],
      dtype='object')

In [13]:
df_rag = df.drop(['job_title', 'location', 'connection', 'relevance'], axis=1)

In [14]:
df_rag.to_csv("../data/rag_hr_data.csv", index=False)